# NER Inference End-to-End — bệnh án thô → JSON 5 type

Ráp 3 nhánh: **A** (encoder + thác ICD/LLM → TRIỆU_CHỨNG/CHẨN_ĐOÁN) ·
**B** (luật + LLM vá → xét nghiệm) · **C** (từ điển RxNorm → THUỐC) →
phủ định → khử chồng lấn → JSON.

**Đầu vào cần add:** Dataset chứa `ner_encoder/` (model đã train, F1 0.818).
Code + KB lấy tươi từ git để tránh bản cũ. `ENABLE_LLM=True` để bật bậc 3 + vá nhánh B.

In [ ]:
# Cell 1 — cài đặt (KHÔNG cài seqeval: vỡ trên Python 3.12)
import os, sys, glob, json, re, subprocess, types
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')  # tránh CUDA re-init khi fork
ENABLE_LLM = True   # False: chỉ KB+luật (nhanh, không cần GPU LLM). True: đủ 3 bậc.
pkgs = ['pyvi'] + (['vllm'] if ENABLE_LLM else [])
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs])
print('cài xong | ENABLE_LLM =', ENABLE_LLM)

In [ ]:
# Cell 2 — tự dò model (dataset) + code & KB (git, tránh bản cũ)
def find_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    hits = [h for h in hits if os.path.isdir(h)]
    return hits[0] if hits else None

MODEL_DIR = find_dir('ner_encoder')
assert MODEL_DIR, "KHÔNG thấy ner_encoder trong /kaggle/input — add Dataset chứa model."
assert os.path.exists(f'{MODEL_DIR}/model.safetensors'), f'{MODEL_DIR} thiếu model.safetensors'
print('MODEL_DIR =', MODEL_DIR)

# Code + KB: ưu tiên git (mới nhất), fallback về fakeer trong dataset
ROOT = None
if not os.path.exists('fakeer'):
    r = subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'],
                       capture_output=True, text=True)
if os.path.exists('fakeer/src'):
    ROOT = 'fakeer'
else:
    ROOT = os.path.dirname(os.path.dirname(find_dir('src') or ''))  # fallback dataset
assert ROOT and os.path.exists(f'{ROOT}/src'), "KHÔNG thấy code (src/)."
sys.path.insert(0, f'{ROOT}/src')

def kb(name):
    p = glob.glob(f'{ROOT}/kb/{name}') or glob.glob(f'/kaggle/input/**/kb/{name}', recursive=True)
    assert p, f'thiếu KB {name}'
    return p[0]
ICD  = kb('icd10_vi_full.csv'); RXN = kb('rxnorm_merged.csv'); INN = kb('inn_usan.csv')
print('ROOT =', ROOT, '| KB ok')

## ⬇️ DÁN BỆNH ÁN VÀO ĐÂY

In [ ]:
TEXT = """Bệnh nhân nam 17 tuổi, vào viện vì nặng mặt, tiểu ít.
Hội chứng thận hư:
Ure: 6,4 mmol/l; Creatinin: 79 micromol/l
Không có suy thận. Không có thiếu máu: HC: 4,49 T/l
Chẩn đoán: Viêm cầu thận mạn - Hội chứng thận hư
Medrol 16mg x 3 viên, uống 8h sáng
Furosemid 40 mg x 1 viên, uống sáng."""
print(len(TEXT), 'ký tự')

In [ ]:
# Cell 4 — NHÁNH B (xét nghiệm, luật) + NHÁNH C (thuốc, từ điển)
from branch_b_lab_tests import extract_lab_pairs, lab_va_candidates
from branch_c_drugs import DrugMatcher

lab_ents = extract_lab_pairs(TEXT)                    # TÊN_XN + KẾT_QUẢ_XN (cấu trúc rõ)
drug_ents = DrugMatcher(RXN, INN).extract_drugs(TEXT) # THUỐC

print(f'Nhánh B (luật): {len(lab_ents)} thực thể xét nghiệm')
print(f'Nhánh C: {len(drug_ents)} thuốc:', [e["text"] for e in drug_ents])
assert all(TEXT[e['start']:e['end']] == e['text'] for e in lab_ents + drug_ents), 'offset lệch'


In [ ]:
# Cell 5 — NHÁNH A bước 1: encoder BIO → span SYM_DIS
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification
from utils.text_alignment import segment_with_map

# DeBERTa-v2 JIT-compile 1 kernel fused (make_log_bucket_position) qua nvrtc;
# trên Kaggle nvrtc hỏng -> "libnvrtc-builtins.so.13.0". Tắt fuser cho chắc.
torch._C._jit_set_texpr_fuser_enabled(False)
try: torch._C._jit_set_nvfuser_enabled(False)
except Exception: pass

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
mdl.eval()
# Encoder chạy CPU: doc ngắn, vài giây, né HẲN lỗi nvrtc trên GPU. vLLM (nhánh
# LLM) vẫn dùng GPU riêng, không ảnh hưởng.
DEV = 'cpu'; mdl.to(DEV)
ID2LAB = {int(k): v for k, v in mdl.config.id2label.items()}

def encoder_spans(text):
    words, spans, ok = segment_with_map(text)
    assert ok, "ánh xạ offset PyVi HỎNG — dừng"
    if not words:
        return []
    # cửa sổ trượt theo TỪ (bệnh án dài > 256 token); giữ nhãn từ cửa sổ có
    # từ nằm xa mép nhất (nhiều ngữ cảnh nhất)
    W, S = 120, 100
    best_lab = [None] * len(words)
    best_center = [-1] * len(words)
    i = 0
    while i < len(words):
        chunk = words[i:i+W]
        enc = tok(chunk, is_split_into_words=True, return_tensors='pt',
                  truncation=True, max_length=256).to(DEV)
        with torch.no_grad():
            pred = mdl(**enc).logits.argmax(-1)[0].cpu().numpy()
        wids = enc.word_ids(0)
        prev = None
        for pos, wid in enumerate(wids):
            if wid is None or wid == prev:
                prev = wid; continue
            prev = wid
            gi = i + wid
            center = min(wid, len(chunk) - 1 - wid)   # khoảng cách tới mép cửa sổ
            if center > best_center[gi]:
                best_center[gi] = center; best_lab[gi] = ID2LAB[int(pred[pos])]
        if i + W >= len(words): break
        i += S
    # gộp B/I liền kề thành span, quy về offset ký tự gốc
    out, a = [], None
    def close(b):
        if a is None: return
        s, e = spans[a][0], spans[b][1]
        out.append({'text': text[s:e], 'start': s, 'end': e, 'score': 0.9,
                    'source': 'encoder'})
    for k, lab in enumerate(best_lab + ['O']):
        lab = lab or 'O'
        if lab == 'B-SYM_DIS':
            if a is not None: close(k-1)
            a = k
        elif lab == 'I-SYM_DIS':
            if a is None: a = k
        else:
            if a is not None: close(k-1); a = None
    return out

sym_spans = encoder_spans(TEXT)
print(f'Nhánh A: {len(sym_spans)} span SYM_DIS:', [e["text"] for e in sym_spans])
assert all(TEXT[e['start']:e['end']] == e['text'] for e in sym_spans), 'offset lệch'


In [ ]:
# Cell 6 — nạp LLM Qwen (nếu ENABLE_LLM): dùng cho bậc 3 nhánh A + vá nhánh B
llm = None
if ENABLE_LLM:
    from llm_choice_classifier import ChoiceClassifier
    LLM_MODEL = 'Qwen/Qwen3-8B'   # ≤9B, self-host. OOM thì đổi Qwen/Qwen2.5-7B-Instruct
    # dùng chung 1 engine vLLM: khởi tạo qua ChoiceClassifier của nhánh B,
    # cascade tier3 sẽ tái dùng cùng model.
    print('sẽ nạp LLM khi gọi (lười nạp để tiết kiệm VRAM)')
else:
    print('ENABLE_LLM=False → bậc 3 mặc định TRIỆU_CHỨNG, bỏ vá nhánh B')


In [ ]:
# Cell 7 — NHÁNH A bước 2: thác tách TRIỆU_CHỨNG/CHẨN_ĐOÁN + NHÁNH B vá LLM
#
# TRIỆT ĐỂ: cascade_classifier import `sentence_transformers`, mà bản mới của nó
# import torchcodec (audio/video) cần libnvrtc.so.13 — Kaggle không có -> vỡ.
# Thay vì hack/pin phiên bản, ta LOẠI HẲN sentence_transformers khỏi runtime:
# cắm một shim mean-pooling dùng transformers thuần (Kaggle có sẵn, đã chạy tốt
# ở cell 5). cascade `from sentence_transformers import SentenceTransformer` sẽ
# nhận lớp này. Không torchcodec, không phụ thuộc phiên bản.
import sys as _sys, types as _types, torch as _torch, numpy as _np
from transformers import AutoTokenizer as _AT, AutoModel as _AM

class _MeanPoolST:
    def __init__(self, name, **_kw):
        self._tok = _AT.from_pretrained(name)
        self._mdl = _AM.from_pretrained(name).eval()   # CPU: né nvrtc như encoder
    def encode(self, texts, convert_to_numpy=True, show_progress_bar=False,
               batch_size=64, **_kw):
        if isinstance(texts, str): texts = [texts]
        out = []
        with _torch.no_grad():
            for i in range(0, len(texts), batch_size):
                b = texts[i:i+batch_size]
                enc = self._tok(b, padding=True, truncation=True, max_length=64,
                                return_tensors='pt')
                h = self._mdl(**enc).last_hidden_state
                m = enc['attention_mask'].unsqueeze(-1).float()
                emb = (h*m).sum(1) / m.sum(1).clamp(min=1e-9)      # mean pooling
                emb = _torch.nn.functional.normalize(emb, p=2, dim=1)  # -> dot = cosine
                out.append(emb.cpu().numpy())
        return _np.vstack(out) if out else _np.zeros((0, 768), dtype='float32')

_st = _types.ModuleType('sentence_transformers')
_st.SentenceTransformer = _MeanPoolST
_sys.modules['sentence_transformers'] = _st

from cascade_classifier import CascadeClassifier

def context_of(text, s, e, win=80):
    a = max(0, s-win); b = min(len(text), e+win)
    return text[a:b].replace('\n', ' ')

# --- thác: bậc 1+2 bằng KB (luôn chạy), bậc 3 bằng LLM (nếu bật) ---
cas = CascadeClassifier(icd_path=ICD, threshold=0.93)

spans_in = [{'text': e['text'], 'context': context_of(TEXT, e['start'], e['end']),
             'start': e['start'], 'end': e['end']} for e in sym_spans]

# KB-only TRƯỚC: luôn ra kết quả kể cả khi LLM lỗi (CUDA init, OOM...).
classified = cas.classify_batch(spans_in, llm_classifier=None)
lab_va = []

# LLM bọc try/except: chết thì BỎ QUA, pipeline vẫn ra JSON với KB+luật.
if ENABLE_LLM and sym_spans:
    try:
        from llm_classifier import LLMClassifier
        llm_client = LLMClassifier(LLM_MODEL)
        classified = cas.classify_batch(spans_in, llm_classifier=llm_client)
        from llm_choice_classifier import ChoiceClassifier
        cands = lab_va_candidates(TEXT, lab_ents)
        if cands:
            clf = ChoiceClassifier(
                model_name=LLM_MODEL,
                labels={'A': ('TÊN_XÉT_NGHIỆM', 'tên xét nghiệm hoặc chỉ số'),
                        'B': ('KẾT_QUẢ_XÉT_NGHIỆM', 'giá trị đo được, gồm số và đơn vị'),
                        'C': ('KHÔNG_PHẢI', 'không thuộc hai loại trên')},
                document=TEXT,
                task_instruction='Bạn là bác sĩ. Chọn đúng một nhãn cho cụm từ trích từ bệnh án.')
            for r in clf.classify_candidates(cands):
                if r['type'] != 'KHÔNG_PHẢI':
                    lab_va.append({'text': r['text'], 'start': r['start'], 'end': r['end'],
                                   'type': r['type'], 'score': 0.6, 'source': 'llm'})
        print(f'LLM OK | nhánh B vá thêm: {len(lab_va)}')
    except Exception as _e:
        print('⚠️ LLM bỏ qua (dùng KB+luật):', type(_e).__name__, str(_e)[:120])

sym_final = []
for e, c in zip(sym_spans, classified):
    src = 'encoder+llm' if c.get('tier','') == 'tier3_llm' else 'encoder+kb'
    sym_final.append({**e, 'type': c['type'], 'score': c.get('confidence', 0.7), 'source': src})
print('sau thác:', [(e['text'], e['type']) for e in sym_final])


In [ ]:
# Cell 8 — gộp 3 nhánh + phủ định + khử chồng lấn → JSON
from negation_detector import NegationDetector
from utils.overlap_resolver import select_non_overlapping

alle = sym_final + lab_ents + lab_va + drug_ents
# khử chồng lấn (QHĐ, điểm cực đại) — mỗi ký tự ≤ 1 thực thể
kept = select_non_overlapping(alle)

# phủ định (2 bước: annotate rồi đọc)
neg = NegationDetector()
kept = neg.annotate_negation(kept, TEXT)
for e in kept:
    e['assertion'] = neg.get_assertion_status(e)

kept.sort(key=lambda x: x['start'])
result = {'text': TEXT, 'entities': [
    {'text': e['text'], 'type': e['type'], 'start': e['start'], 'end': e['end'],
     'score': round(float(e.get('score', 1.0)), 3), 'source': e.get('source','rule'),
     'negated': bool(e.get('negated', False)), 'assertion': e.get('assertion','affirmed')}
    for e in kept]}

# BẤT BIẾN bắt buộc
T = result['text']
assert all(e['text'] == T[e['start']:e['end']] for e in result['entities']), 'span KHÔNG nguyên văn'
o = result['entities']
assert all(o[i]['end'] <= o[i+1]['start'] for i in range(len(o)-1)), 'CHỒNG LẤN'
VALID = {'TRIỆU_CHỨNG','CHẨN_ĐOÁN','THUỐC','TÊN_XÉT_NGHIỆM','KẾT_QUẢ_XÉT_NGHIỆM'}
assert all(e['type'] in VALID for e in o), 'type LẠ'

json.dump(result, open('/kaggle/working/ner_output.json','w',encoding='utf-8'),
          ensure_ascii=False, indent=2)

import collections
c = collections.Counter(e['type'] for e in o)
print(f"{'='*56}\n{len(o)} thực thể | {dict(c)}")
for e in o:
    neg = ' [phủ định]' if e['negated'] else ''
    print(f"  {e['type']:20s} {e['text'][:40]!r:42s} ({e['source']}){neg}")
print('\n=> /kaggle/working/ner_output.json')